# Project Report — Determination of the Point Group and Visualization of Molecular Symmetry Elements

**Programming project in chemistry**  
**Topic:** Automatic determination of a molecule’s point group and visualization of its symmetry axes/planes.

> This notebook combines both the scientific report of the project, the explanation of the code, and executable cells for testing or presenting the program.

## 1. Introduction

Molecular symmetry is a fundamental concept in chemistry. It allows molecules to be classified according to their **point group**, that is, the set of symmetry operations that leave the molecule unchanged: rotations, symmetry planes, inversion center, improper axes, etc.

This classification is important because it plays a role in several fields:

- infrared and Raman spectroscopy;
- polarity analysis;
- chirality identification.

## 2. Project Objectives

The main objective of the project is to create a program capable of:

1. asking the user for the IUPAC name of a molecule;
2. automatically retrieving the chemical information of this molecule;
3. generating a 3D molecular structure;
4. determining its point group;
5. displaying symmetry operations as matrices;
6. providing an interactive 3D visualization of symmetry axes, planes, and centers.

The project is divided into two complementary parts.

## 3. General Principle of the Program

The program follows a processing pipeline in several steps:

1. **User input**: the user provides the IUPAC name of the molecule.
2. **PubChem search**: the program retrieves the corresponding compound, its CID, formula, molar mass, and SMILES.
3. **Molecular construction**: the SMILES is converted into a molecular object using RDKit.
4. **Hydrogen addition**: hydrogens are explicitly added to obtain a more realistic geometry.

## 4. Part 1 — Determination of the Point Group

The first script uses several specialized libraries:

- **PubChemPy** to query the PubChem database;
- **RDKit** to build a molecule from the SMILES and generate a 3D geometry;
- **NumPy** to manipulate coordinates;
- **pymatgen** to analyze molecular symmetry and determine the point group.

The choice of these libraries is important: instead of completely rewriting symmetry algorithms, the project relies on validated scientific tools.

In [ ]:
# Cellule optionnelle : installation des dépendances si nécessaire
# À décommenter dans un environnement neuf.

# !pip install pubchempy rdkit pymatgen numpy streamlit py3Dmol stmol

In [ ]:
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from pymatgen.core import Molecule as PmgMol
from pymatgen.symmetry.analyzer import PointGroupAnalyzer

In [ ]:
def analyser_molecule(iupac_name):
    """
    Analyse une molécule à partir de son nom IUPAC.
    Retourne un dictionnaire contenant les informations principales,
    les coordonnées atomiques, le groupe ponctuel et les opérations de symétrie.
    """
    iupac_name = iupac_name.strip()
    print(f"Recherche de : {iupac_name}")

    compounds = pcp.get_compounds(iupac_name, "name")
    if not compounds:
        raise ValueError(f"Aucun composé trouvé pour {iupac_name}")

    c = compounds[0]
    smiles = c.canonical_smiles
    formule = c.molecular_formula
    masse = c.molecular_weight
    cid = c.cid

    print(f"CID : {cid}")
    print(f"Formule : {formule}")
    print(f"Masse : {masse} g/mol")
    print(f"SMILES : {smiles}")

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    result = AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    if result == -1:
        AllChem.EmbedMolecule(mol, AllChem.ETKDG())

    AllChem.MMFFOptimizeMolecule(mol)

    conf = mol.GetConformer()
    symbols = []
    coords = []

    for i, atom in enumerate(mol.GetAtoms()):
        pos = conf.GetAtomPosition(i)
        symbols.append(atom.GetSymbol())
        coords.append([pos.x, pos.y, pos.z])

    coords = np.array(coords)

    print(f"{len(symbols)} atomes positionnés")
    print(f"Shape coords : {coords.shape}")
    print(f"Symboles : {symbols}")

    pmg_mol = PmgMol(symbols, coords)
    analyzer = PointGroupAnalyzer(pmg_mol)

    point_group = str(analyzer.get_pointgroup())
    sym_ops = analyzer.get_symmetry_operations()

    print(f"Groupe ponctuel : {point_group}")
    print(f"Nombre d'opérations : {len(sym_ops)}")

    return {
        "nom": iupac_name,
        "cid": cid,
        "formule": formule,
        "masse": masse,
        "smiles": smiles,
        "symbols": symbols,
        "coords": coords,
        "point_group": point_group,
        "sym_ops": sym_ops,
    }

In [ ]:
# Exemple d'utilisation
# Vous pouvez remplacer "ammonia" par "water", "methane", "benzene", etc.

# resultat = analyser_molecule("ammonia")

In [ ]:
# Affichage des matrices d'opérations de symétrie
# À exécuter après la cellule précédente.

# for i, op in enumerate(resultat["sym_ops"], 1):
#     print(f"\nOpération {i}:")
#     print(op.rotation_matrix)

## 5. Part 2 — Visualization of Symmetry Elements

The second part of the project aims to make the result easier to understand. A point group written as text, for example `C3v`, is useful, but it remains abstract for a student. Visualization makes it possible to connect this result to the real geometry of the molecule.

The visualization script uses:

- **Streamlit** to create a simple interface;
- **py3Dmol** to display the molecule in 3D;
- **stmol** to integrate the visualization into Streamlit.

In [ ]:
import numpy as np

# Ces imports sont nécessaires uniquement si l'on lance l'interface Streamlit.
# import streamlit as st
# import py3Dmol
# from stmol import showmol

In [ ]:
def construire_xyz(molecule_data):
    """
    Construit un texte au format XYZ à partir d'un dictionnaire molecule_data.
    Ce format est directement lisible par py3Dmol.
    """
    atomes = molecule_data["atomes"]
    lignes = [str(len(atomes)), molecule_data.get("nom", "molecule")]

    for a in atomes:
        lignes.append(f"{a['element']}  {a['x']:.4f}  {a['y']:.4f}  {a['z']:.4f}")

    return "\n".join(lignes)

In [ ]:
def deduire_proprietes(molecule_data):
    """
    Déduit quelques propriétés chimiques simples à partir du groupe ponctuel.
    Ces règles sont approximatives et servent surtout à l'interprétation pédagogique.
    """
    import re

    pg = molecule_data.get("point_group", "C1")

    groupes_chiraux = {"C1", "C2", "C3", "C4", "C5", "C6", "D2", "D3", "D4", "D5", "D6", "T", "O", "I"}
    chiral = molecule_data.get("chiral", pg in groupes_chiraux)
    polaire = molecule_data.get("polar", bool(re.match(r"^C\d+(v)?$", pg)))
    ir = molecule_data.get("ir_active", True)
    raman = molecule_data.get("raman_active", True)

    if molecule_data.get("inversion", False):
        ir_txt = "Partiel, selon la règle d'exclusion"
        raman_txt = "Partiel, selon la règle d'exclusion"
    else:
        ir_txt = "Oui" if ir else "Non"
        raman_txt = "Oui" if raman else "Non"

    return chiral, polaire, ir_txt, raman_txt

In [ ]:
# Exemple de dictionnaire de visualisation pour NH3

molecule_data = {
    "nom": "ammonia",
    "formule": "NH3",
    "point_group": "C3v",
    "chiral": False,
    "polar": True,
    "ir_active": True,
    "raman_active": True,

    "atomes": [
        {"element": "N", "x": 0.000, "y": 0.000, "z": 0.000},
        {"element": "H", "x": 0.939, "y": 0.000, "z": -0.333},
        {"element": "H", "x": -0.470, "y": 0.813, "z": -0.333},
        {"element": "H", "x": -0.470, "y": -0.813, "z": -0.333},
    ],

    "liaisons": [[0, 1], [0, 2], [0, 3]],

    "axes": [
        {"direction": [0, 0, 1], "ordre": 3, "label": "C3"},
    ],

    "plans": [
        {"normale": [1, 0, 0], "type": "σv", "label": "σv"},
        {"normale": [0, 1, 0], "type": "σv", "label": "σv'"},
        {"normale": [0.5, 0.866, 0], "type": "σv", "label": "σv''"},
    ],

    "inversion": False,
}

print(construire_xyz(molecule_data))
print(deduire_proprietes(molecule_data))

## 6. Defense of the Project Idea

The project has a double interest: **scientific** and **educational**.

From a scientific point of view, the program shows how chemical data from a public database can be transformed into a 3D structure and then analyzed using symmetry tools. It therefore connects several important notions of computational chemistry: SMILES representation, conformer generation, geometry optimization, atomic coordinates, symmetry analysis, and visualization.

## 7. Comparison with Existing Programs

There are already several software programs capable of analyzing molecular symmetry, for example:

- **Avogadro**, which allows the construction and visualization of molecules;
- **ChemCraft** or **GaussView**, often used with quantum chemistry calculations;
- **Molden**, for visualizing structures and orbitals;
- **pymatgen**, which already includes point group analysis tools;
- some modules associated with **Gaussian**, **ORCA**, or other quantum chemistry software.

## 8. Why We Did Not Simply Use Existing Software

We could have directly used already available software, but that would have limited the educational value of the project. By building the processing pipeline ourselves, we learned how to:

- query a chemical database;
- manipulate SMILES strings;
- generate a molecular geometry;
- extract atomic coordinates;
- use a symmetry library;
- structure a program into several modules.

## 9. Limitations of the Code

Our program has several important limitations.

### 9.1 Dependence on PubChem

The program depends on PubChem searches. If the name provided by the user is not recognized, if several compounds are possible, or if the returned result is not the expected one, the analysis may be incorrect.

### 9.2 Approximate 3D Geometry

The 3D structure is generated by RDKit from the SMILES and then optimized using a force field. This geometry is often correct for visualization, but it remains approximate.

## 10. Possible Improvements

Several improvements could be added:

- provide a selection when PubChem returns multiple compounds;
- automatically export the results to the dictionary used for visualization;
- automatically extract axes and planes from symmetry operations;
- add a single interface combining search, calculation, and visualization;
- allow the import of `.xyz`, `.mol`, `.sdf`, or `.pdb` files;
- compare the result with reference databases.

## 11. Conclusion

This project shows how programming can help solve a classical chemistry problem: determining the point group of a molecule. The program retrieves chemical information, generates a three-dimensional structure, analyzes symmetry, and provides an interactive visualization.

Even though the code has limitations, especially regarding geometric precision and the full automation of visualization, it constitutes a solid basis for an educational and scientific tool.

## 12. Appendix — Launching the Streamlit Interface

To launch the visualization interface from a terminal, place the `visualisation.py` file in the project folder, then run:

```bash
streamlit run visualisation.py
```

The notebook is mainly intended for the report and code explanations. The Streamlit application remains more suitable for the complete interactive interface.